This section goes to data preprocessing.

Ensure to install java, CICFlowMeter Maven

In [1]:
import pandas as pd

file_path = "../dataset/processed/adware_experimental.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["class"].value_counts())

Dataset shape: (359984, 80)

Class distribution:
class
Benign             236368
Adware             120071
General Malware      3545
Name: count, dtype: int64


In [2]:
constant_features = [
    "furg_cnt",
    "burg_cnt",
    "flow_urg",
    "flow_cwr",
    "flow_ece"
]

df = df.drop(columns=constant_features)

print("Updated dataset shape:", df.shape)
print("\nRemaining constant features:")
print(df.nunique()[df.nunique() == 1])

Updated dataset shape: (359984, 75)

Remaining constant features:
Series([], dtype: int64)


In [3]:
X = df.drop(columns=["class"])
y = df["class"]

print("Feature dataset shape:", X.shape)
print("Target dataset shape:", y.shape)

print("\nTarget classes:")
print(y.value_counts())

Feature dataset shape: (359984, 74)
Target dataset shape: (359984,)

Target classes:
class
Benign             236368
Adware             120071
General Malware      3545
Name: count, dtype: int64


In [4]:
print("Feature data types:")
print(X.dtypes.value_counts())

print("\nNon-numeric columns:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

print("\nMissing values:")
print(X.isnull().sum().sum())

Feature data types:
int64      50
float64    24
Name: count, dtype: int64

Non-numeric columns:
[]

Missing values:
0


In [5]:
import numpy as np

infinite_counts = np.isinf(X).sum()

print("Total infinite values:", infinite_counts.sum())

print("\nFeatures containing infinite values:")
print(infinite_counts[infinite_counts > 0])

Total infinite values: 0

Features containing infinite values:
Series([], dtype: int64)


In [6]:
negative_counts = (X < 0).sum()

print("Features containing negative values:")
print(negative_counts[negative_counts > 0].sort_values(ascending=False))

print("\nTotal negative feature values:", negative_counts.sum())

Features containing negative values:
max_biat                   209189
min_biat                   209189
min_idle                   204558
min_active                 204558
max_active                 204558
max_idle                   204558
min_fiat                   101194
max_fiat                   101194
Init_Win_bytes_backward     98938
max_bpktl                   98938
min_bpktl                   98938
bVarianceDataBytes           4277
min_fpktl                    3769
Init_Win_bytes_forward       3769
max_fpktl                    3769
min_flowiat                  3314
max_flowiat                  2095
dtype: int64

Total negative feature values: 1756805


In [7]:
sentinel_features = [
    "min_fpktl",
    "max_fpktl",
    "min_bpktl",
    "max_bpktl",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for feature in sentinel_features:
    negative_values = X.loc[X[feature] < 0, feature]

    print(f"\n{feature}")
    print("Negative records:", len(negative_values))
    print("Unique negative values:", negative_values.unique()[:10])


min_fpktl
Negative records: 3769
Unique negative values: [-1]

max_fpktl
Negative records: 3769
Unique negative values: [-1]

min_bpktl
Negative records: 98938
Unique negative values: [-1]

max_bpktl
Negative records: 98938
Unique negative values: [-1]

Init_Win_bytes_forward
Negative records: 3769
Unique negative values: [-1]

Init_Win_bytes_backward
Negative records: 98938
Unique negative values: [-1]


In [8]:
active_idle_features = [
    "min_active",
    "max_active",
    "min_idle",
    "max_idle"
]

for feature in active_idle_features:
    negative_values = X.loc[X[feature] < 0, feature]

    print(f"\n{feature}")
    print("Negative records:", len(negative_values))
    print("Unique negative values:", negative_values.unique()[:10])

print("\nAre all four features negative in exactly the same records?")

reference = X["min_active"] < 0

for feature in active_idle_features[1:]:
    print(feature, (reference == (X[feature] < 0)).all())


min_active
Negative records: 204558
Unique negative values: [-1]

max_active
Negative records: 204558
Unique negative values: [-1]

min_idle
Negative records: 204558
Unique negative values: [-1]

max_idle
Negative records: 204558
Unique negative values: [-1]

Are all four features negative in exactly the same records?
max_active True
min_idle True
max_idle True


In [9]:
flow_iat_features = [
    "min_flowiat",
    "max_flowiat"
]

for feature in flow_iat_features:
    negative_values = X.loc[X[feature] < 0, feature]

    print(f"\n{feature}")
    print("Negative records:", len(negative_values))
    print("Minimum negative value:", negative_values.min())
    print("Maximum negative value:", negative_values.max())
    print("Unique negative values:", negative_values.nunique())
    print("\nMost common negative values:")
    print(negative_values.value_counts().head(10))


min_flowiat
Negative records: 3314
Minimum negative value: -181
Maximum negative value: -1
Unique negative values: 63

Most common negative values:
min_flowiat
-1     2130
-7       58
-8       48
-18      42
-14      41
-20      39
-6       39
-21      39
-9       38
-19      37
Name: count, dtype: int64

max_flowiat
Negative records: 2095
Minimum negative value: -1
Maximum negative value: -1
Unique negative values: 1

Most common negative values:
max_flowiat
-1    2095
Name: count, dtype: int64


In [10]:
negative_iat = X[X["min_flowiat"] < 0].copy()

packet_columns = [
    "total_fpackets",
    "total_bpackets"
]

print("Negative min_flowiat records:", len(negative_iat))

print("\nPacket count statistics:")
print(negative_iat[packet_columns].describe())

print("\nRecords with fewer than 2 packets in both directions:")
print(
    (
        (negative_iat["total_fpackets"] < 2) &
        (negative_iat["total_bpackets"] < 2)
    ).sum()
)

print("\nRecords with at least 2 packets in both directions:")
print(
    (
        (negative_iat["total_fpackets"] >= 2) &
        (negative_iat["total_bpackets"] >= 2)
    ).sum()
)

Negative min_flowiat records: 3314

Packet count statistics:
       total_fpackets  total_bpackets
count     3314.000000     3314.000000
mean       607.951418     1346.894689
std       2315.521857     4605.250510
min          0.000000        0.000000
25%          1.000000        0.000000
50%          1.000000        0.000000
75%        366.500000      808.250000
max      48255.000000    74768.000000

Records with fewer than 2 packets in both directions:
2095

Records with at least 2 packets in both directions:
1219


In [11]:
negative_variance = X.loc[
    X["bVarianceDataBytes"] < 0,
    "bVarianceDataBytes"
]

print("Negative records:", len(negative_variance))
print("Minimum value:", negative_variance.min())
print("Maximum value:", negative_variance.max())
print("Unique negative values:", negative_variance.nunique())

print("\nMost common negative values:")
print(negative_variance.value_counts().head(15))

Negative records: 4277
Minimum value: -1.0
Maximum value: -3.29e-05
Unique negative values: 27

Most common negative values:
bVarianceDataBytes
-0.200000    1837
-1.000000    1244
-0.166667     639
-0.142857     227
-0.500000     160
-0.250000      65
-0.333333      53
-0.125000      20
-0.111111       5
-0.066667       4
-0.020833       3
-0.083333       3
-0.050000       2
-0.052632       2
-0.071429       1
Name: count, dtype: int64


In [12]:
negative_variance_df = df.loc[
    X["bVarianceDataBytes"] < 0
].copy()

print("Negative variance records:", len(negative_variance_df))

print("\nBackward packet statistics:")
print(
    negative_variance_df["total_bpackets"].describe()
)

print("\nRecords with 0 backward packets:")
print(
    (negative_variance_df["total_bpackets"] == 0).sum()
)

print("\nRecords with 1 backward packet:")
print(
    (negative_variance_df["total_bpackets"] == 1).sum()
)

print("\nRecords with more than 1 backward packet:")
print(
    (negative_variance_df["total_bpackets"] > 1).sum()
)

Negative variance records: 4277

Backward packet statistics:
count     4277.000000
mean        13.642740
std        474.653795
min          2.000000
25%          2.000000
50%          6.000000
75%          6.000000
max      30393.000000
Name: total_bpackets, dtype: float64

Records with 0 backward packets:
0

Records with 1 backward packet:
0

Records with more than 1 backward packet:
4277


In [13]:
negative_summary = pd.DataFrame({
    "Feature": [
        "Packet length and TCP window features",
        "Active/idle features",
        "min_flowiat",
        "max_flowiat",
        "bVarianceDataBytes"
    ],
    "Finding": [
        "Negative values are exclusively -1",
        "Negative values are exclusively -1 and occur in identical records",
        "Contains multiple negative values (-181 to -1)",
        "Negative values are exclusively -1",
        "Contains 27 distinct negative values"
    ],
    "Decision": [
        "Preserve",
        "Preserve",
        "Preserve for further investigation",
        "Preserve",
        "Preserve for further investigation"
    ]
})

display(negative_summary)

,Feature,Finding,Decision
0,Packet length and TCP window features,Negative values are exclusively -1,Preserve
1,Active/idle features,Negative values are exclusively -1 and occur i...,Preserve
2,min_flowiat,Contains multiple negative values (-181 to -1),Preserve for further investigation
3,max_flowiat,Negative values are exclusively -1,Preserve
4,bVarianceDataBytes,Contains 27 distinct negative values,Preserve for further investigation


In [14]:
packet_iat_features = [
    "min_fiat",
    "max_fiat",
    "min_biat",
    "max_biat"
]

for feature in packet_iat_features:
    negative_values = X.loc[X[feature] < 0, feature]

    print(f"\n{feature}")
    print("Negative records:", len(negative_values))
    print("Minimum negative value:", negative_values.min())
    print("Maximum negative value:", negative_values.max())
    print("Unique negative values:", negative_values.nunique())
    print("Most common negative values:")
    print(negative_values.value_counts().head(10))


min_fiat
Negative records: 101194
Minimum negative value: -1
Maximum negative value: -1
Unique negative values: 1
Most common negative values:
min_fiat
-1    101194
Name: count, dtype: int64

max_fiat
Negative records: 101194
Minimum negative value: -1
Maximum negative value: -1
Unique negative values: 1
Most common negative values:
max_fiat
-1    101194
Name: count, dtype: int64

min_biat
Negative records: 209189
Minimum negative value: -1
Maximum negative value: -1
Unique negative values: 1
Most common negative values:
min_biat
-1    209189
Name: count, dtype: int64

max_biat
Negative records: 209189
Minimum negative value: -1
Maximum negative value: -1
Unique negative values: 1
Most common negative values:
max_biat
-1    209189
Name: count, dtype: int64


In [15]:
checks = {
    "min_fiat": X["total_fpackets"] < 2,
    "max_fiat": X["total_fpackets"] < 2,
    "min_biat": X["total_bpackets"] < 2,
    "max_biat": X["total_bpackets"] < 2
}

for feature, expected_condition in checks.items():
    negative_condition = X[feature] < 0

    print(f"\n{feature}")
    print("Negative records:", negative_condition.sum())
    print("Matching expected condition:",
          (negative_condition == expected_condition).sum())
    print("Non-matching records:",
          (negative_condition != expected_condition).sum())


min_fiat
Negative records: 101194
Matching expected condition: 359984
Non-matching records: 0

max_fiat
Negative records: 101194
Matching expected condition: 359984
Non-matching records: 0

min_biat
Negative records: 209189
Matching expected condition: 359984
Non-matching records: 0

max_biat
Negative records: 209189
Matching expected condition: 359984
Non-matching records: 0


In [16]:
negative_min = X["min_flowiat"] < 0
negative_max = X["max_flowiat"] < 0

print("Negative min_flowiat:", negative_min.sum())
print("Negative max_flowiat:", negative_max.sum())

print("\nBoth features negative:",
      (negative_min & negative_max).sum())

print("Only min_flowiat negative:",
      (negative_min & ~negative_max).sum())

print("Only max_flowiat negative:",
      (~negative_min & negative_max).sum())

Negative min_flowiat: 3314
Negative max_flowiat: 2095

Both features negative: 2095
Only min_flowiat negative: 1219
Only max_flowiat negative: 0


In [17]:
negative_decisions = pd.DataFrame([
    {
        "Feature Group": "Packet length and TCP window",
        "Finding": "Negative values exclusively -1",
        "Decision": "Preserve"
    },
    {
        "Feature Group": "Active/idle",
        "Finding": "All four features share identical -1 positions",
        "Decision": "Preserve"
    },
    {
        "Feature Group": "Forward/backward packet IAT",
        "Finding": "Negative values exclusively -1; 100% match insufficient packet counts",
        "Decision": "Preserve"
    },
    {
        "Feature Group": "Flow IAT",
        "Finding": "min_flowiat has multiple negative values; max_flowiat contains only -1",
        "Decision": "Preserve; investigate further"
    },
    {
        "Feature Group": "bVarianceDataBytes",
        "Finding": "4,277 negative values despite more than one backward packet",
        "Decision": "Preserve; investigate feature calculation"
    }
])

display(negative_decisions)

negative_decisions.to_csv(
    "../results/negative_value_investigation.csv",
    index=False
)

print("Negative-value investigation saved!")

,Feature Group,Finding,Decision
0,Packet length and TCP window,Negative values exclusively -1,Preserve
1,Active/idle,All four features share identical -1 positions,Preserve
2,Forward/backward packet IAT,Negative values exclusively -1; 100% match ins...,Preserve
3,Flow IAT,min_flowiat has multiple negative values; max_...,Preserve; investigate further
4,bVarianceDataBytes,"4,277 negative values despite more than one ba...",Preserve; investigate feature calculation


Negative-value investigation saved!


In [18]:
feature_summary = X.describe().T

feature_summary = feature_summary[
    ["min", "max", "mean", "std"]
]

display(feature_summary)

,min,max,mean,std
duration,0.0,4.431076e+10,3.832135e+07,2.505609e+08
total_fpackets,0.0,4.825500e+04,1.098323e+01,2.306649e+02
total_bpackets,0.0,7.476800e+04,1.820524e+01,4.628200e+02
total_fpktl,0.0,4.049644e+07,1.483293e+03,1.091076e+05
total_bpktl,0.0,1.039222e+08,2.115955e+04,6.391036e+05
...,...,...,...,...
Init_Win_bytes_forward,-1.0,4.194240e+06,1.607731e+06,1.975637e+06
Init_Win_bytes_backward,-1.0,4.194240e+06,5.408350e+05,8.071800e+05
RRT_samples_clnt,0.0,7.452400e+04,1.700234e+01,4.607899e+02
Act_data_pkt_forward,0.0,4.825500e+04,1.097682e+01,2.306342e+02


In [19]:
skewness = X.skew().sort_values(ascending=False)

print("Top 15 positively skewed features:")
display(skewness.head(15))

print("\nTop 15 negatively skewed features:")
display(skewness.tail(15))

Top 15 positively skewed features:


FFNEPD                  501.846814
sflow_fbytes            417.603989
fAvgPacketsPerBulk      402.161133
total_fpktl             330.237292
std_fiat                308.242994
std_biat                266.567583
std_flowiat             207.309468
max_biat                133.278497
total_biat              131.068509
min_biat                122.908071
sflow_fpacket           118.454980
mean_biat               111.267860
sflow_bbytes             98.358160
sflow_bpacket            97.346428
Act_data_pkt_forward     86.789077
dtype: float64


Top 15 negatively skewed features:


mean_bpktl                 2.337589
bVarianceDataBytes         2.330422
avgPacketSize              2.256583
mean_flowpktl              2.256583
std_fpktl                  1.799746
std_bpktl                  1.735182
max_fpktl                  1.692046
max_bpktl                  1.578143
Init_Win_bytes_backward    1.527301
flow_syn                   1.389715
max_flowpktl               1.173027
std_flowpktl               1.165900
Init_Win_bytes_forward     0.533310
flow_fin                   0.294717
min_seg_size_forward      -1.645103
dtype: float64

In [20]:
print("FFNEPD statistics:")
print(X["FFNEPD"].describe())

print("\nUnique values:")
print("Total unique values:", X["FFNEPD"].nunique())

print("\nMost common values:")
print(X["FFNEPD"].value_counts().head(15))

FFNEPD statistics:
count    359984.000000
mean          2.525384
std           4.010521
min           2.000000
25%           2.000000
50%           2.000000
75%           3.000000
max        2269.000000
Name: FFNEPD, dtype: float64

Unique values:
Total unique values: 59

Most common values:
FFNEPD
2     242361
3      93217
4      10959
5       4280
6       3062
7       2364
9       1247
8       1203
17       370
11       299
10       146
19       122
13        44
12        38
21        31
Name: count, dtype: int64


In [21]:
print("FFNEPD percentiles:")

print(
    X["FFNEPD"].quantile([
        0.90,
        0.95,
        0.99,
        0.995,
        0.999,
        1.00
    ])
)

print("\nRecords above selected thresholds:")

for threshold in [10, 20, 50, 100, 500, 1000]:
    count = (X["FFNEPD"] > threshold).sum()

    print(
        f"Greater than {threshold}: "
        f"{count:,} records "
        f"({count / len(X) * 100:.4f}%)"
    )

FFNEPD percentiles:
0.900       3.0
0.950       4.0
0.990       8.0
0.995       9.0
0.999      17.0
1.000    2269.0
Name: FFNEPD, dtype: float64

Records above selected thresholds:
Greater than 10: 1,145 records (0.3181%)
Greater than 20: 165 records (0.0458%)
Greater than 50: 16 records (0.0044%)
Greater than 100: 2 records (0.0006%)
Greater than 500: 1 records (0.0003%)
Greater than 1000: 1 records (0.0003%)


In [22]:
ffnepd_by_class = df.groupby("class")["FFNEPD"].agg(
    ["count", "mean", "median", "max"]
)

display(ffnepd_by_class)

,count,mean,median,max
class,,,,
Adware,120071,2.472437,2.0,75
Benign,236368,2.549816,2.0,2269
General Malware,3545,2.689704,2.0,10


In [23]:
outlier_summary = pd.DataFrame({
    "99th_percentile": X.quantile(0.99),
    "maximum": X.max()
})

outlier_summary["max_to_99th_ratio"] = (
    outlier_summary["maximum"] /
    outlier_summary["99th_percentile"].replace(0, np.nan)
)

outlier_summary = outlier_summary.sort_values(
    "max_to_99th_ratio",
    ascending=False
)

display(outlier_summary.head(15))

,99th_percentile,maximum,max_to_99th_ratio
sflow_fbytes,4616.00,4.049644e+07,8773.059575
sflow_bbytes,24973.17,1.039222e+08,4161.352964
total_fpktl,9903.00,4.049644e+07,4089.310613
sflow_bpacket,27.00,7.476800e+04,2769.185185
sflow_fpacket,23.00,4.825500e+04,2098.043478
total_bpktl,76872.98,1.039222e+08,1351.868693
RRT_samples_clnt,62.00,7.452400e+04,1202.000000
total_bpackets,70.00,7.476800e+04,1068.114286
total_bhlen,1400.00,1.495360e+06,1068.114286
min_biat,10981045.52,1.154102e+10,1050.994245


In [24]:
class_distribution = y.value_counts().to_frame("Count")

class_distribution["Percentage"] = (
    y.value_counts(normalize=True) * 100
).round(2)

display(class_distribution)

,Count,Percentage
class,,
Benign,236368,65.66
Adware,120071,33.35
General Malware,3545,0.98


In [25]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [26]:
duplicate_features = df.drop(columns=["class"]).duplicated().sum()

print("Duplicate feature combinations:", duplicate_features)

Duplicate feature combinations: 0


In [27]:
print("Unique class labels:")
print(y.unique())

print("\nNumber of unique classes:", y.nunique())

Unique class labels:
<StringArray>
['Benign', 'Adware', 'General Malware']
Length: 3, dtype: str

Number of unique classes: 3


In [28]:
print("Dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nNumber of features:", X.shape[1])
print("Number of records:", X.shape[0])

Dataset shape: (359984, 75)
Feature matrix shape: (359984, 74)
Target vector shape: (359984,)

Number of features: 74
Number of records: 359984


In [29]:
output_path = "../dataset/processed/adware_cleaned.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Location:", output_path)


Cleaned dataset saved successfully!
Location: ../dataset/processed/adware_cleaned.csv


In [31]:
import os

original_path = "../dataset/original"

for root, dirs, files in os.walk(original_path):
    level = root.replace(original_path, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

original/
    TotalFeatures-ISCXFlowMeter.csv
    Adware Pcaps/
        1.pcap
        10.pcap
        11.pcap
        12.pcap
        13.pcap
        14.pcap
        15.pcap
        16.pcap
        17.pcap
        18.pcap
    Benign Pcaps/
        1.pcap
        10.pcap
        100.pcap
        101.pcap
        102.pcap
        103.pcap
        104.pcap
        105.pcap
        106.pcap
        107.pcap
    General Malware Pcaps/
        1.pcap
        2.pcap
        3.pcap
        4.pcap
        5.pcap
        6.pcap
        7.pcap


In [32]:
original_csv_path = "../dataset/original/TotalFeatures-ISCXFlowMeter.csv"

original_sample = pd.read_csv(original_csv_path, nrows=5)

print("Original CSV columns:")
print(original_sample.columns.tolist())

print("\nFirst 5 records:")
display(original_sample)

Original CSV columns:
['duration', 'total_fpackets', 'total_bpackets', 'total_fpktl', 'total_bpktl', 'min_fpktl', 'min_bpktl', 'max_fpktl', 'max_bpktl', 'mean_fpktl', 'mean_bpktl', 'std_fpktl', 'std_bpktl', 'total_fiat', 'total_biat', 'min_fiat', 'min_biat', 'max_fiat', 'max_biat', 'mean_fiat', 'mean_biat', 'std_fiat', 'std_biat', 'fpsh_cnt', 'bpsh_cnt', 'furg_cnt', 'burg_cnt', 'total_fhlen', 'total_bhlen', 'fPktsPerSecond', 'bPktsPerSecond', 'flowPktsPerSecond', 'flowBytesPerSecond', 'min_flowpktl', 'max_flowpktl', 'mean_flowpktl', 'std_flowpktl', 'min_flowiat', 'max_flowiat', 'mean_flowiat', 'std_flowiat', 'flow_fin', 'flow_syn', 'flow_rst', 'flow_psh', 'flow_ack', 'flow_urg', 'flow_cwr', 'flow_ece', 'downUpRatio', 'avgPacketSize', 'fAvgSegmentSize', 'fHeaderBytes', 'fAvgBytesPerBulk', 'fAvgPacketsPerBulk', 'fAvgBulkRate', 'bVarianceDataBytes', 'bAvgSegmentSize', 'bAvgBytesPerBulk', 'bAvgPacketsPerBulk', 'bAvgBulkRate', 'sflow_fpacket', 'sflow_fbytes', 'sflow_bpacket', 'sflow_bbytes'

,duration,total_fpackets,total_bpackets,total_fpktl,total_bpktl,min_fpktl,min_bpktl,max_fpktl,max_bpktl,mean_fpktl,...,mean_idle,max_idle,std_idle,FFNEPD,Init_Win_bytes_forward,Init_Win_bytes_backward,RRT_samples_clnt,Act_data_pkt_forward,min_seg_size_forward,calss
0,1020586,668,1641,35692,2276876,52,52,679,1390,53.431138,...,0,-1,0,2,4194240,1853440,1640,668,32,benign
1,80794,1,1,75,124,75,124,75,124,75.000000,...,0,-1,0,2,0,0,0,1,0,benign
2,998,3,0,187,0,52,-1,83,-1,62.333333,...,0,-1,0,4,101888,-1,0,3,32,benign
3,189868,9,9,1448,6200,52,52,706,1390,160.888889,...,0,-1,0,2,4194240,2722560,8,9,32,benign
4,110577,4,6,528,1422,52,52,331,1005,132.000000,...,0,-1,0,2,155136,31232,5,4,32,benign


In [33]:
pcap_folders = {
    "Adware": "../dataset/original/Adware Pcaps",
    "Benign": "../dataset/original/Benign Pcaps",
    "General Malware": "../dataset/original/General Malware Pcaps"
}

for label, folder in pcap_folders.items():
    pcap_files = [
        file for file in os.listdir(folder)
        if file.lower().endswith(".pcap")
    ]

    print(f"{label}: {len(pcap_files)} PCAP files")

Adware: 62 PCAP files
Benign: 173 PCAP files
General Malware: 7 PCAP files


In [34]:
for label, folder in pcap_folders.items():
    print(f"\n{label}")

    total_size = 0

    for file in os.listdir(folder):
        if file.lower().endswith(".pcap"):
            file_path = os.path.join(folder, file)
            total_size += os.path.getsize(file_path)

    print(f"Total size: {total_size / (1024**3):.2f} GB")


Adware
Total size: 6.70 GB

Benign
Total size: 3.40 GB

General Malware
Total size: 0.18 GB


In [44]:
import pandas as pd

original_csv_path = "../dataset/original/TotalFeatures-ISCXFlowMeter.csv"

original_info = pd.read_csv(original_csv_path, nrows=0)

print("CSV filename:", os.path.basename(original_csv_path))
print("Number of columns:", len(original_info.columns))
print("First column:", original_info.columns[0])
print("Last column:", original_info.columns[-1])

CSV filename: TotalFeatures-ISCXFlowMeter.csv
Number of columns: 80
First column: duration
Last column: calss


In [46]:
print("X and y have matching indices:", X.index.equals(y.index))
print("X contains missing values:", X.isnull().sum().sum())
print("y contains missing values:", y.isnull().sum())

X and y have matching indices: True
X contains missing values: 0
y contains missing values: 0


In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training features: (287987, 74)
Testing features: (71997, 74)

Training labels: (287987,)
Testing labels: (71997,)


In [48]:
print("Training class distribution:")
print(
    y_train.value_counts(normalize=True).mul(100).round(2)
)

print("\nTesting class distribution:")
print(
    y_test.value_counts(normalize=True).mul(100).round(2)
)

Training class distribution:
class
Benign             65.66
Adware             33.35
General Malware     0.98
Name: proportion, dtype: float64

Testing class distribution:
class
Benign             65.66
Adware             33.35
General Malware     0.98
Name: proportion, dtype: float64


In [49]:
X_train.assign(class_label=y_train).to_csv(
    "../dataset/processed/train.csv", index=False
)

X_test.assign(class_label=y_test).to_csv(
    "../dataset/processed/test.csv", index=False
)

print("Training and testing datasets saved successfully!")

Training and testing datasets saved successfully!
